# Step 2: Passive Membrane Analysis
This notebook analyzes the passive properties of the selected neuron using hyperpolarizing sweeps.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import linregress
from allensdk.core.cell_types_cache import CellTypesCache

# 1. Load data
ctc = CellTypesCache(manifest_file='../data/raw/manifest.json')
specimen_id = 485909730
data_set = ctc.get_ephys_data(specimen_id)
all_sweeps = ctc.get_ephys_sweeps(specimen_id)


## 2. Select sweeps
We select hyperpolarizing sweeps (24-29) to avoid active conductances.

In [ ]:
target_sweeps = [24, 25, 26, 27, 28, 29]
print(f"Analyzing sweeps: {target_sweeps}")


## 3. Estimate resting potential
Calculate mean of the pre-stimulus baseline.

In [ ]:
def estimate_resting_potential(time, voltage, stim_start_s, pre_window=0.5):
    mask = (time >= stim_start_s - pre_window) & (time < stim_start_s)
    return np.mean(voltage[mask]), np.std(voltage[mask])


## 4. Estimate input resistance
Use steady-state voltage deflection relative to baseline.

In [ ]:
def estimate_steady_state(time, voltage, stim_end_s, tail_window=0.2):
    mask = (time > stim_end_s - tail_window) & (time <= stim_end_s)
    return np.mean(voltage[mask])


## 5. Estimate tau
Fit exponential decay to initial transient.

In [ ]:
def exp_decay(t, v_inf, v_0, tau):
    return v_inf + (v_0 - v_inf) * np.exp(-t / tau)

def fit_tau(time, voltage, stim_start, v_0, v_ss):
    mask = (time > stim_start) & (time < stim_start + 0.1)
    t_fit = time[mask] - stim_start
    v_fit = voltage[mask]
    
    p0 = [v_ss, v_0, 0.015]
    bounds = ([-np.inf, -np.inf, 0.001], [np.inf, np.inf, 0.2])
    popt, _ = curve_fit(exp_decay, t_fit, v_fit, p0=p0, bounds=bounds)
    return popt[2]


## 6. Derive capacitance & Run Analysis

In [ ]:
# Run analysis on sweeps
results = []
currents_A, delta_vs_V = [], []

for sw in target_sweeps:
    sweep_data = data_set.get_sweep(sw)
    meta = next((s for s in all_sweeps if s['sweep_number'] == sw))
    time = np.arange(len(sweep_data['response'])) / float(sweep_data['sampling_rate'])
    v = sweep_data['response']
    
    start_s = meta['stimulus_start_time']
    end_s = start_s + meta['stimulus_duration']
    amp_A = meta['stimulus_absolute_amplitude'] * 1e-12
    
    v_rest, _ = estimate_resting_potential(time, v, start_s)
    v_ss = estimate_steady_state(time, v, end_s)
    delta_v = v_ss - v_rest
    
    tau_s = fit_tau(time, v, start_s, v_rest, v_ss)
    r_m = delta_v / amp_A
    c_m = tau_s / r_m
    
    currents_A.append(amp_A)
    delta_vs_V.append(delta_v)
    results.append({'sweep': sw, 'V_rest': v_rest, 'R_m': r_m, 'tau': tau_s, 'C_m': c_m})

pd.DataFrame(results)


## 7. Simulate passive model
Using explicit Euler.

In [ ]:
def simulate_passive(time, stimulus, v_rest, r_m, tau_m):
    dt = time[1] - time[0]
    v_sim = np.zeros_like(time)
    v_sim[0] = v_rest
    for i in range(1, len(time)):
        dv = (-(v_sim[i-1] - v_rest) + r_m * stimulus[i-1]) / tau_m
        v_sim[i] = v_sim[i-1] + dt * dv
    return v_sim


## 8. Compare model with experiment & 9. Calculate error

In [ ]:
# Use robust parameters
res = linregress(np.array(currents_A)*1e12, np.array(delta_vs_V)*1e3)
r_m_robust = res.slope * 1e9 # Ohms
tau_robust = np.median([r['tau'] for r in results])
v_rest_robust = np.mean([r['V_rest'] for r in results])

# Sweep 24 for validation
sw24_data = data_set.get_sweep(24)
time24 = np.arange(len(sw24_data['response'])) / float(sw24_data['sampling_rate'])
v_sim = simulate_passive(time24, sw24_data['stimulus'], v_rest_robust, r_m_robust, tau_robust)

mask = (time24 >= 1.02) & (time24 <= 2.02)
rmse = np.sqrt(np.mean((sw24_data['response'][mask] - v_sim[mask])**2))
print(f"RMSE (Active window): {rmse*1000:.3f} mV")

plt.plot(time24, sw24_data['response']*1000, label='Data')
plt.plot(time24, v_sim*1000, label='Model', linestyle='--')
plt.xlim(0.8, 2.5)
plt.legend()
plt.show()


## 10. Sensitivity analysis
Vary tau by 50%, 100%, 200%.

In [ ]:
for factor in [0.5, 1.0, 2.0]:
    v_sim_sens = simulate_passive(time24, sw24_data['stimulus'], v_rest_robust, r_m_robust, tau_robust * factor)
    plt.plot(time24, v_sim_sens*1000, label=f'Model tau={factor*100:.0f}%')
plt.xlim(0.9, 1.3)
plt.legend()
plt.show()


## 11. Summarize findings
The linear passive membrane model closely captures the subthreshold voltage response. A sharp transient difference is seen if tau is mis-estimated. Action potentials are NOT captured by this model.